# DLENT-RT — Ablations, round 1 (single factors), small-cluster regime

Kash: apply the ablations one at a time, find which ones matter most, then decide the cumulative order. This notebook
covers the **single-factor round only** (no cumulative runs).

**Regime**: the committed `configs/default.yaml` + ONLY the capacity block of `configs/regime_small_cluster.yaml`
(both from git HEAD, `scripts/regime_config.py`); 30k trace; theory model with ξ = 0.001, γ = 1.0, δ = 1/T³.
**Seeds**: only the thinning seed varies (`online.thinning_seed_offset` = 2, 3, 4, 5, 6); Phase 0, H, the stream and the
bids are fixed — the thinning draws are the model's only online randomness.

**Ablations** (each a `cfg.ablation` flag or an existing `constants` key, default = the theory model):
* **A1 thinning** (only the thinning cap p_U; the tests are untouched): A1a p_U = 1; A1b δ = 1/T inside p_U's ε_D;
  A1c ε_D = 0 inside p_U (cap 1 − ξ).
* **A2a ξ** ∈ {0.01, 0.05} (γ = 1); **A2b γ** ∈ {0.3, 0.1} (ξ = 0.001): forced constants, global by nature.
* **A3 fixed grid**: ReInit keeps Phase 0's valuation grid instead of refitting it.

This notebook only READS the cache of `scripts/ablation_runner.py` (`reports/week30/ablations/`). Build:
`python notebooks/src/build.py 05_ablations_small_cluster --execute`.

In [1]:
import subprocess, sys, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dlent_rt").is_dir())
ADIR = ROOT / "reports" / "week30" / "ablations"
FIGS = ROOT / "reports" / "week30" / "figs_ablations"
FIGS.mkdir(parents=True, exist_ok=True)
for f in ("runs.csv", "summary.csv", "references.json"):
    if not (ADIR / f).is_file():
        raise FileNotFoundError(f"{ADIR / f} missing: run `dlent_venv/bin/python scripts/ablation_runner.py run` first")
RUNS = pd.read_csv(ADIR / "runs.csv")
SUM = pd.read_csv(ADIR / "summary.csv", index_col=0)
REF = json.loads((ADIR / "references.json").read_text())
_git = lambda *a: subprocess.check_output(["git", "-C", str(ROOT), *a], text=True).strip()
NB_COMMIT = _git("rev-parse", "HEAD")
REGIME = "configs/regime_small_cluster.yaml"
print(f"NB05 RUN MARKER  notebook built at commit {NB_COMMIT}; runs at commit(s) {sorted(RUNS['commit'].unique())}")
print(f"configs/default.yaml blob: runs {sorted(RUNS['default blob'].unique())}, HEAD {_git('rev-parse', 'HEAD:configs/default.yaml')}")
print(f"{REGIME} blob: runs {sorted(RUNS['regime blob'].unique())}, HEAD {_git('rev-parse', f'HEAD:{REGIME}')}")
print(f"capacity {sorted(RUNS['capacity'].unique())}  s_C {sorted(RUNS['s_C'].round(6).unique())}  "
      f"tau_0 {sorted(RUNS['tau_0'].unique())}")
print(f"{len(RUNS)} runs: {RUNS.ablation.nunique()} ablations x seed offsets {sorted(RUNS['seed offset'].unique())}")
ORDER = list(SUM.index)
JM, CEIL, SP = REF["job MILP $"], REF["feasible ceiling $"], REF["static Phase 0 $"]


def savefig(fig, name):
    fig.savefig(FIGS / f"{name}.png", dpi=200, bbox_inches="tight")

NB05 RUN MARKER  notebook built at commit eba0c633ca91d6734edee66910deffb108cfa85c; runs at commit(s) ['eba0c633ca91d6734edee66910deffb108cfa85c']
configs/default.yaml blob: runs ['68e7bd76b57c376cae6d74139f11cc456559520b'], HEAD 68e7bd76b57c376cae6d74139f11cc456559520b
configs/regime_small_cluster.yaml blob: runs ['a0d29b5313ea41184961aa085366bc988c5dd7d0'], HEAD a0d29b5313ea41184961aa085366bc988c5dd7d0
capacity ['[0.4014237441083392, 0.2616208305922613]']  s_C [np.float64(228.184283)]  tau_0 ['[0.0, 0.020386]']
45 runs: 9 ablations x seed offsets [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


## 1. Baseline check and references

The baseline (all flags off, thinning seed offset 2) must reproduce the theory run of `small_cluster_nofilter.md`:
DLENT-RT = 40,892.14 $. References (baseline config): job MILP, feasible ceiling, static Phase 0, static + thinning per
seed (the same thinning draws as DLENT-RT with that seed).

In [2]:
b2 = RUNS[(RUNS.ablation == "baseline") & (RUNS["seed offset"] == 2)].iloc[0]
print(f"baseline, offset 2: R = {b2['R $']:,.2f} $  (expected 40,892.14)")
assert round(b2["R $"], 2) == 40_892.14
assert RUNS["s_C"].round(6).nunique() == 1 and RUNS["tau_0"].nunique() == 1 and RUNS["capacity"].nunique() == 1
refs = pd.Series({k: v for k, v in REF.items() if k.endswith("$")}, name="$")
display(refs.to_frame().style.format("{:,.2f}"))
print("job MILP:", REF["job MILP status"], "gap", REF["job MILP gap"])

baseline, offset 2: R = 40,892.14 $  (expected 40,892.14)


,$
job MILP $,"120,742.02"
feasible ceiling $,"141,126.62"
static Phase 0 $,"44,716.94"


job MILP: Optimization terminated successfully. (HiGHS Status 7: Optimal) gap 9.913961929749156e-05


## 2–5. One section per ablation family

Per level: the constants it changes (seed-independent), the reward over the 5 seeds (mean ± std), the paired
difference vs the baseline with the same thinning seed (mean and 95% t-interval, 4 degrees of freedom), and the
reward as % of the job MILP, of the feasible ceiling and of static Phase 0. Figure: reward per seed and level, with
the baseline per seed as a line.

In [3]:
CONST = ["xi", "gamma", "eps_D (tests)", "eps_D (thinning)", "p_U cap", "p_U at phase-1 start", "p_U at t - tau = d_max",
         "eta(d_max)", "Test 2 factor 1-eps_D-xi", "eps_Test2(d_max)", "eps_Test3(d_max)", "Test 1 margin xi*c CPU",
         "Test 1 margin xi*c RAM"]
RES = ["R mean $", "R std $", "paired delta mean $", "paired delta 95% lo $", "paired delta 95% hi $",
       "beyond seed noise (95% CI excludes 0)", "% of job MILP (mean)", "% of feasible ceiling (mean)",
       "% of static Phase 0 (mean)"]
base_seed = RUNS[RUNS.ablation == "baseline"].set_index("seed offset")["R $"]


def family(fam, title, fig_name):
    names = ["baseline"] + [n for n in ORDER if SUM.loc[n, "family"] == fam]
    print(f"=== {title}")
    display(SUM.loc[names, ["level"] + CONST].T)
    display(SUM.loc[names, ["level"] + RES].style.format(precision=2))
    fig, ax = plt.subplots(figsize=(9, 3.6))
    seeds = sorted(RUNS["seed offset"].unique())
    w = 0.8 / len(names)
    for i, n in enumerate(names):
        g = RUNS[RUNS.ablation == n].set_index("seed offset").reindex(seeds)
        ax.bar(np.arange(len(seeds)) + (i - (len(names) - 1) / 2) * w, g["R $"], w,
               label=f"{n} ({SUM.loc[n, 'level']})", color="dimgray" if n == "baseline" else None)
    ax.axhline(JM, color="k", ls="--", lw=0.8, label=f"job MILP {JM:,.0f}")
    ax.axhline(SP, color="darkorange", ls=":", lw=1.0, label=f"static Phase 0 {SP:,.0f}")
    ax.set_xticks(np.arange(len(seeds)), [f"offset {s}" for s in seeds])
    ax.set_ylabel("R $ (DLENT-RT)")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout(); savefig(fig, fig_name); plt.show()


family("A1 thinning", "A1 — thinning relaxed (p_U only)", "A1_thinning")

=== A1 — thinning relaxed (p_U only)


ablation,baseline,A1a_thinning_off,A1b_delta_1_T,A1c_epsD_0
level,theory,p_U = 1,delta = 1/T in p_U,eps_D = 0 in p_U
xi,0.001,0.001,0.001,0.001
gamma,1.0,1.0,1.0,1.0
eps_D (tests),0.50222,0.50222,0.50222,0.50222
eps_D (thinning),0.50222,0.50222,0.273609,0.0
p_U cap,0.665016,1.0,0.784385,0.999
p_U at phase-1 start,0.186502,1.0,0.194817,0.205798
p_U at t - tau = d_max,0.637874,1.0,0.746901,0.938981
eta(d_max),0.031959,0.031959,0.031959,0.031959
Test 2 factor 1-eps_D-xi,0.49678,0.49678,0.49678,0.49678


,level,R mean $,R std $,paired delta mean $,paired delta 95% lo $,paired delta 95% hi $,beyond seed noise (95% CI excludes 0),% of job MILP (mean),% of feasible ceiling (mean),% of static Phase 0 (mean)
ablation,,,,,,,,,,
baseline,theory,39739.06,987.18,0.00,nan,nan,False,32.91,28.16,88.87
A1a_thinning_off,p_U = 1,58621.35,0.00,18882.30,17656.55,20108.05,True,48.55,41.54,131.09
A1b_delta_1_T,delta = 1/T in p_U,46675.28,1632.94,6936.23,4743.38,9129.08,True,38.66,33.07,104.38
A1c_epsD_0,eps_D = 0 in p_U,57655.08,1159.33,17916.02,16489.10,19342.95,True,47.75,40.85,128.93


/var/folders/5d/ncqx2wnn67s3dbzcdsz504nw0000gn/T/ipykernel_84509/3722362046.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig(fig, fig_name); plt.show()


In [4]:
family("A2a xi", "A2a — xi_override (gamma = 1)", "A2a_xi")

=== A2a — xi_override (gamma = 1)


ablation,baseline,A2a_xi_0.01,A2a_xi_0.05
level,theory,xi = 0.01,xi = 0.05
xi,0.001,0.01,0.05
gamma,1.0,1.0,1.0
eps_D (tests),0.50222,2.179196,7.261907
eps_D (thinning),0.50222,2.179196,7.261907
p_U cap,0.665016,0.311399,0.114986
p_U at phase-1 start,0.186502,0.140755,0.078407
p_U at t - tau = d_max,0.637874,0.305262,0.114103
eta(d_max),0.031959,0.031959,0.031959
Test 2 factor 1-eps_D-xi,0.49678,-1.189196,-6.311907


,level,R mean $,R std $,paired delta mean $,paired delta 95% lo $,paired delta 95% hi $,beyond seed noise (95% CI excludes 0),% of job MILP (mean),% of feasible ceiling (mean),% of static Phase 0 (mean)
ablation,,,,,,,,,,
baseline,theory,39739.06,987.18,0.00,nan,nan,False,32.91,28.16,88.87
A2a_xi_0.01,xi = 0.01,19399.89,1745.56,-20339.17,-22647.42,-18030.91,True,16.07,13.75,43.38
A2a_xi_0.05,xi = 0.05,7294.52,551.60,-32444.53,-33941.32,-30947.74,True,6.04,5.17,16.31


/var/folders/5d/ncqx2wnn67s3dbzcdsz504nw0000gn/T/ipykernel_84509/3722362046.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig(fig, fig_name); plt.show()


In [5]:
family("A2b gamma", "A2b — gamma_override (xi = 0.001)", "A2b_gamma")

=== A2b — gamma_override (xi = 0.001)


ablation,baseline,A2b_gamma_0.3,A2b_gamma_0.1
level,theory,gamma = 0.3,gamma = 0.1
xi,0.001,0.001,0.001
gamma,1.0,0.3,0.1
eps_D (tests),0.50222,0.50222,0.50222
eps_D (thinning),0.50222,0.50222,0.50222
p_U cap,0.665016,0.665016,0.665016
p_U at phase-1 start,0.186502,0.375775,0.52923
p_U at t - tau = d_max,0.637874,0.656634,0.662198
eta(d_max),0.031959,0.009588,0.003196
Test 2 factor 1-eps_D-xi,0.49678,0.49678,0.49678


,level,R mean $,R std $,paired delta mean $,paired delta 95% lo $,paired delta 95% hi $,beyond seed noise (95% CI excludes 0),% of job MILP (mean),% of feasible ceiling (mean),% of static Phase 0 (mean)
ablation,,,,,,,,,,
baseline,theory,39739.06,987.18,0.00,nan,nan,False,32.91,28.16,88.87
A2b_gamma_0.3,gamma = 0.3,40689.63,1100.13,950.58,-955.58,2856.73,False,33.70,28.83,90.99
A2b_gamma_0.1,gamma = 0.1,40700.30,3307.49,961.25,-1990.23,3912.73,False,33.71,28.84,91.02


/var/folders/5d/ncqx2wnn67s3dbzcdsz504nw0000gn/T/ipykernel_84509/3722362046.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig(fig, fig_name); plt.show()


In [6]:
family("A3 grid", "A3 — fixed grid across phases (Phase 0 grid kept)", "A3_fixed_grid")

=== A3 — fixed grid across phases (Phase 0 grid kept)


ablation,baseline,A3_fixed_grid
level,theory,Phase 0 grid kept
xi,0.001,0.001
gamma,1.0,1.0
eps_D (tests),0.50222,0.50222
eps_D (thinning),0.50222,0.50222
p_U cap,0.665016,0.665016
p_U at phase-1 start,0.186502,0.186502
p_U at t - tau = d_max,0.637874,0.637874
eta(d_max),0.031959,0.031959
Test 2 factor 1-eps_D-xi,0.49678,0.49678


,level,R mean $,R std $,paired delta mean $,paired delta 95% lo $,paired delta 95% hi $,beyond seed noise (95% CI excludes 0),% of job MILP (mean),% of feasible ceiling (mean),% of static Phase 0 (mean)
ablation,,,,,,,,,,
baseline,theory,39739.06,987.18,0.00,nan,nan,False,32.91,28.16,88.87
A3_fixed_grid,Phase 0 grid kept,41344.72,1646.85,1605.66,-57.50,3268.83,False,34.24,29.30,92.46


/var/folders/5d/ncqx2wnn67s3dbzcdsz504nw0000gn/T/ipykernel_84509/3722362046.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig(fig, fig_name); plt.show()


## 6. Ranking of the single-factor effects

Paired difference vs the baseline (same thinning seed), mean over the 5 seeds with its 95% t-interval, in $ and as % of
the job MILP. Bars whose interval excludes 0 are larger than the seed noise (dark); the others are not (light).

In [7]:
RANK = SUM.drop(index="baseline").sort_values("paired delta mean $", ascending=False)
display(RANK[["family", "level", "paired delta mean $", "paired delta 95% lo $", "paired delta 95% hi $",
              "paired delta % of job MILP", "beyond seed noise (95% CI excludes 0)", "% of job MILP (mean)",
              "% of static Phase 0 (mean)"]].style.format(precision=2))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
y = np.arange(len(RANK))[::-1]
err = np.vstack([RANK["paired delta mean $"] - RANK["paired delta 95% lo $"],
                 RANK["paired delta 95% hi $"] - RANK["paired delta mean $"]])
col = ["teal" if b else "lightsteelblue" for b in RANK["beyond seed noise (95% CI excludes 0)"]]
axes[0].barh(y, RANK["paired delta mean $"], xerr=err, color=col, capsize=3)
axes[0].set_yticks(y, [f"{n} ({RANK.loc[n, 'level']})" for n in RANK.index], fontsize=8)
axes[0].axvline(0, color="k", lw=0.7); axes[0].set_xlabel("paired Δ R vs baseline [$] (95% CI)")
axes[1].barh(y, RANK["paired delta % of job MILP"], xerr=100 * err / JM, color=col, capsize=3)
axes[1].set_yticks(y, [""] * len(y)); axes[1].axvline(0, color="k", lw=0.7)
axes[1].set_xlabel("paired Δ R vs baseline [% of job MILP]")
fig.suptitle("Single-factor ablation effects (dark = beyond the seed noise)", fontsize=10)
plt.tight_layout(); savefig(fig, "ranking"); plt.show()

,family,level,paired delta mean $,paired delta 95% lo $,paired delta 95% hi $,paired delta % of job MILP,beyond seed noise (95% CI excludes 0),% of job MILP (mean),% of static Phase 0 (mean)
ablation,,,,,,,,,
A1a_thinning_off,A1 thinning,p_U = 1,18882.30,17656.55,20108.05,15.64,True,48.55,131.09
A1c_epsD_0,A1 thinning,eps_D = 0 in p_U,17916.02,16489.10,19342.95,14.84,True,47.75,128.93
A1b_delta_1_T,A1 thinning,delta = 1/T in p_U,6936.23,4743.38,9129.08,5.74,True,38.66,104.38
A3_fixed_grid,A3 grid,Phase 0 grid kept,1605.66,-57.50,3268.83,1.33,False,34.24,92.46
A2b_gamma_0.1,A2b gamma,gamma = 0.1,961.25,-1990.23,3912.73,0.80,False,33.71,91.02
A2b_gamma_0.3,A2b gamma,gamma = 0.3,950.58,-955.58,2856.73,0.79,False,33.70,90.99
A2a_xi_0.01,A2a xi,xi = 0.01,-20339.17,-22647.42,-18030.91,-16.85,True,16.07,43.38
A2a_xi_0.05,A2a xi,xi = 0.05,-32444.53,-33941.32,-30947.74,-26.87,True,6.04,16.31


/var/folders/5d/ncqx2wnn67s3dbzcdsz504nw0000gn/T/ipykernel_84509/2677636651.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig(fig, "ranking"); plt.show()


## 7. Micro-benchmarks per ablation (means over the 5 seeds)

Phase end reasons, phases with τ > 0, whether Test 1 fires (number of runs out of 5), admitted jobs, thinning and
physical rejects, and the reward split (customer value / profit / carbon).

In [8]:
MICRO = ["level", "phases (mean)", "end timeout (mean)", "end test1 (mean)", "end test2 (mean)", "end test3 (mean)",
         "end horizon (mean)", "phases with tau > 0 (mean)", "Test 1 fires (runs)", "admitted (mean)",
         "thinning rejects (mean)", "physical rejects (mean)", "customer value $ (mean)", "profit $ (mean)",
         "carbon $ (mean)"]
display(SUM[MICRO].style.format(precision=1))
fig, ax = plt.subplots(figsize=(12, 3.8))
x = np.arange(len(SUM))
for j, (k, c) in enumerate((("admitted (mean)", "seagreen"), ("thinning rejects (mean)", "darkorange"),
                            ("physical rejects (mean)", "indianred"))):
    ax.bar(x + (j - 1) * 0.27, SUM[k], 0.27, color=c, label=k)
ax.set_xticks(x, SUM.index, rotation=25, ha="right", fontsize=8); ax.legend(fontsize=8)
ax.set_title("Admissions and rejections per ablation (means over 5 seeds)", fontsize=10)
plt.tight_layout(); savefig(fig, "micro_rejects"); plt.show()

,level,phases (mean),end timeout (mean),end test1 (mean),end test2 (mean),end test3 (mean),end horizon (mean),phases with tau > 0 (mean),Test 1 fires (runs),admitted (mean),thinning rejects (mean),physical rejects (mean),customer value $ (mean),profit $ (mean),carbon $ (mean)
ablation,,,,,,,,,,,,,,,
baseline,theory,9.4,6.4,1.6,0.4,0.0,1.0,5.8,4,6636.0,3811.2,500.6,37471.3,6197.7,-3930.0
A1a_thinning_off,p_U = 1,10.0,5.0,4.0,0.0,0.0,1.0,7.0,5,9880.0,0.0,1144.0,54641.0,9267.2,-5286.8
A1b_delta_1_T,delta = 1/T in p_U,10.6,6.4,2.6,0.6,0.0,1.0,7.0,5,7736.0,2575.4,698.0,43722.6,7387.1,-4434.3
A1c_epsD_0,eps_D = 0 in p_U,11.2,5.2,5.0,0.0,0.0,1.0,7.2,5,9596.0,326.2,1050.6,53511.5,9329.8,-5186.2
A2a_xi_0.01,xi = 0.01,8.2,5.2,2.0,0.0,0.0,1.0,5.0,4,3240.0,7630.6,134.0,18604.3,2907.8,-2112.3
A2a_xi_0.05,xi = 0.05,8.0,5.6,1.4,0.0,0.0,1.0,4.8,3,1216.6,9705.8,34.4,7207.9,1007.8,-921.1
A2b_gamma_0.3,gamma = 0.3,11.8,7.6,1.4,1.0,0.8,1.0,7.4,4,6668.0,3708.4,503.6,38108.8,6493.1,-3912.2
A2b_gamma_0.1,gamma = 0.1,53.2,9.6,0.4,26.8,15.4,1.0,32.6,2,6491.4,3611.0,474.4,39908.2,4654.8,-3862.7
A3_fixed_grid,Phase 0 grid kept,11.6,7.2,2.8,0.6,0.0,1.0,5.0,5,6757.6,3939.6,547.2,38235.6,7236.0,-4126.9


/var/folders/5d/ncqx2wnn67s3dbzcdsz504nw0000gn/T/ipykernel_84509/2586321340.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); savefig(fig, "micro_rejects"); plt.show()


## 8. Which guarantee each ablation gives up (stated from the code; the paper's proofs are not in the repo)

In [9]:
display(SUM[["level", "voids"]])

,level,voids
ablation,,
baseline,theory,-
A1a_thinning_off,p_U = 1,the Bernoulli thinning p_U = (1 - xi)/(1 + eps...
A1b_delta_1_T,delta = 1/T in p_U,the confidence level of eps_D used by the thin...
A1c_epsD_0,eps_D = 0 in p_U,the eps_D margin of the thinning (cap (1 - xi)...
A2a_xi_0.01,xi = 0.01,xi is forced (not data): a larger xi changes e...
A2a_xi_0.05,xi = 0.05,as xi = 0.01
A2b_gamma_0.3,gamma = 0.3,gamma is forced (not data): a smaller gamma sh...
A2b_gamma_0.1,gamma = 0.1,as gamma = 0.3
A3_fixed_grid,Phase 0 grid kept,ReInit's re-estimation of the valuation grid (...


## 9. All tables as markdown (`reports/week30/ablations/ablations_tables.md`)

In [10]:
def _fmt(v):
    if isinstance(v, (bool, np.bool_)):
        return str(v)
    if isinstance(v, (int, np.integer)):
        return f"{v:,}"
    if isinstance(v, (float, np.floating)):
        return "nan" if not np.isfinite(v) else (f"{v:,.2f}" if abs(v) >= 1 or v == 0 else f"{v:.3g}")
    return str(v)


def md_table(df, index_name):
    head = "| " + index_name + " | " + " | ".join(map(str, df.columns)) + " |"
    sep = "|" + "---|" * (len(df.columns) + 1)
    body = ["| " + " / ".join(map(str, k if isinstance(k, tuple) else (k,))) + " | "
            + " | ".join(_fmt(v) for v in row) + " |" for k, row in zip(df.index, df.itertuples(index=False))]
    return "\n".join([head, sep, *body])


md = ["# Ablations round 1: generated tables (notebooks/05_ablations_small_cluster.ipynb; do not edit by hand)\n",
      f"Notebook commit `{NB_COMMIT}`; runs at commit(s) {sorted(RUNS['commit'].unique())}; regime `{REGIME}`.\n",
      "## References\n", md_table(refs.to_frame(), "reference"), "",
      "## Constants per ablation\n", md_table(SUM[["level"] + CONST], "ablation"), "",
      "## Reward per ablation (5 thinning seeds)\n", md_table(SUM[["level"] + RES], "ablation"), "",
      "## Ranking (paired delta vs baseline)\n",
      md_table(RANK[["level", "paired delta mean $", "paired delta 95% lo $", "paired delta 95% hi $",
                     "paired delta % of job MILP", "beyond seed noise (95% CI excludes 0)"]], "ablation"), "",
      "## Micro-benchmarks (means over 5 seeds)\n", md_table(SUM[MICRO], "ablation"), "",
      "## Guarantee given up\n", md_table(SUM[["level", "voids"]], "ablation"), "",
      "## Every run\n", md_table(RUNS.set_index(["ablation", "seed offset"])[
          ["R $", "customer value $", "profit $", "carbon $", "admitted", "thinning rejects", "physical rejects",
           "phases", "end timeout", "end test1", "end test2", "end test3", "end horizon", "phases with tau > 0",
           "% of job MILP", "% of feasible ceiling", "% of static Phase 0"]], "ablation / seed"), ""]
(ADIR / "ablations_tables.md").write_text("\n".join(md))
print("markdown ->", ADIR / "ablations_tables.md")

markdown -> /Users/francescodangolo/Projects/dlent_rt/reports/week30/ablations/ablations_tables.md
